# Dual-Wavelength CG Optimization

Shared-hologram optimization for two wavelengths using two slmsuite calibration files.

## 1. Imports and Parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from slm_tophat_beam_shaping.dual_wavelength_optimizer import DWCGOptimizer
from slm_tophat_beam_shaping.initial_holograms import curvature_hologram
from slm_tophat_beam_shaping.loss_functions import build_circular_mask
from slm_tophat_beam_shaping.optical_planes import CameraPlane, SLMPlane
from slm_tophat_beam_shaping.propagator import Propagator
from slm_tophat_beam_shaping.target_profiles import apply_psf_smoothing, build_rectangle_target

# Native SLM and compute grid.
full_slm_shape = (1080, 1920)
superpixel_size = 4
slm_shape = (full_slm_shape[0] // superpixel_size, full_slm_shape[1] // superpixel_size)
padding_factor = 2
camera_shape = (slm_shape[0] * padding_factor, slm_shape[1] * padding_factor)

# Shared optical parameters.
slm_pixel_pitch_um = 8.0
camera_pixel_pitch_um = 3.45
focal_length_mm = 200.0

# Two wavelengths and their slmsuite wavefront-calibration files.
wavelength_1_nm = 420.0
wavelength_2_nm = 456.0
slmsuite_calibration_h5_path_1 = PROJECT_ROOT / "10806-SLM-wavefront_superpixel-calibration_00036.h5"
slmsuite_calibration_h5_path_2 = PROJECT_ROOT / "10806-SLM-wavefront_superpixel-calibration_00030.h5"
slmsuite_camera_shape = full_slm_shape
slmsuite_wavefront_r2_threshold = 0.5
slmsuite_remove_background = True
slmsuite_apply_calibration = True
slmsuite_amplitude_key = "amplitude"
slmsuite_phase_key = "phase"

# Same physical target and mask parameters for both wavelengths.
rectangle_width_x_um = 300.0
rectangle_width_y_um = 100.0
psf_sigma_x_um = 10.0
psf_sigma_y_um = 10.0
mask_radius_margin_um = 100.0

# Initial shared hologram parameters.
initial_phase_linear_tilt = 0.0
initial_phase_astigmatism_weight = 0.0
initial_phase_quadratic_curvature = 3.6e-3
initial_phase_linear_angle_rad = np.pi / 4.0
initial_phase_conical_weight = 0.0

# DWCG parameters.
optimizer_maxiter = 100
loss_scale = 1e12
optimize_phase = True
loss_method = "linear"
channel_weight = 0.5
exponential_rate = 1.0


## 2. Build Optical Systems and Targets

In [ ]:
def downsample_by_superpixel(data, superpixel_size):
    data_array = np.asarray(data, dtype=float)
    if data_array.ndim != 2:
        raise ValueError(f"data must be 2D, got shape {data_array.shape}.")
    if data_array.shape[0] % superpixel_size != 0 or data_array.shape[1] % superpixel_size != 0:
        raise ValueError(f"data shape must be divisible by superpixel_size, got data_shape={data_array.shape}, superpixel_size={superpixel_size}.")
    ny = data_array.shape[0] // superpixel_size
    nx = data_array.shape[1] // superpixel_size
    return data_array.reshape(ny, superpixel_size, nx, superpixel_size).mean(axis=(1, 3))


def downsample_phase_by_superpixel(phase, superpixel_size):
    phase_array = np.asarray(phase, dtype=float)
    if phase_array.ndim != 2:
        raise ValueError(f"phase must be 2D, got shape {phase_array.shape}.")
    real_part = downsample_by_superpixel(np.cos(phase_array), superpixel_size)
    imag_part = downsample_by_superpixel(np.sin(phase_array), superpixel_size)
    return np.angle(real_part + 1j * imag_part)


def require_calibration_array(calibration_results, key, expected_shape):
    if key not in calibration_results:
        available_keys = sorted(str(item) for item in calibration_results.keys())
        raise KeyError(f"Calibration result key {key!r} was not found. Available keys: {available_keys}.")
    array = np.asarray(calibration_results[key], dtype=float)
    if array.shape != expected_shape:
        raise ValueError(f"Calibration result {key!r} shape must be {expected_shape}, got {array.shape}.")
    if not np.all(np.isfinite(array)):
        raise ValueError(f"Calibration result {key!r} contains NaN or infinite values.")
    return array


def load_slmsuite_input_beam_and_phase(calibration_h5_path, full_slm_shape, superpixel_size, slm_pixel_pitch_um, wavelength_nm, camera_shape, r2_threshold, remove_background, apply_calibration, amplitude_key, phase_key):
    from slmsuite.hardware.cameras.simulated import SimulatedCamera
    from slmsuite.hardware.cameraslms import FourierSLM
    from slmsuite.hardware.slms.simulated import SimulatedSLM

    slm_size_xy = (int(full_slm_shape[1]), int(full_slm_shape[0]))
    camera_size_xy = (int(camera_shape[1]), int(camera_shape[0]))
    slm = SimulatedSLM(slm_size_xy, pitch_um=(slm_pixel_pitch_um, slm_pixel_pitch_um), wav_um=wavelength_nm / 1000.0)
    camera = SimulatedCamera(slm, resolution=camera_size_xy)
    fourier_slm = FourierSLM(camera, slm)
    fourier_slm.load_calibration("wavefront_superpixel", file_path=calibration_h5_path)
    calibration_results = fourier_slm.wavefront_calibration_superpixel_process(
        plot=False,
        r2_threshold=r2_threshold,
        remove_background=remove_background,
        apply=apply_calibration,
    )
    amplitude_full = require_calibration_array(calibration_results, amplitude_key, full_slm_shape)
    phase_full = require_calibration_array(calibration_results, phase_key, full_slm_shape)
    amplitude = downsample_by_superpixel(np.clip(amplitude_full, 0.0, None), superpixel_size)
    phase = downsample_phase_by_superpixel(phase_full, superpixel_size)
    if np.max(amplitude) <= 0:
        raise ValueError("Loaded slmsuite amplitude contains no positive values after downsampling.")
    return amplitude / np.max(amplitude), np.mod(phase, 2.0 * np.pi), calibration_results


def build_channel(wavelength_nm, calibration_h5_path):
    input_amplitude, input_phase, calibration_results = load_slmsuite_input_beam_and_phase(
        calibration_h5_path=calibration_h5_path,
        full_slm_shape=full_slm_shape,
        superpixel_size=superpixel_size,
        slm_pixel_pitch_um=slm_pixel_pitch_um,
        wavelength_nm=wavelength_nm,
        camera_shape=slmsuite_camera_shape,
        r2_threshold=slmsuite_wavefront_r2_threshold,
        remove_background=slmsuite_remove_background,
        apply_calibration=slmsuite_apply_calibration,
        amplitude_key=slmsuite_amplitude_key,
        phase_key=slmsuite_phase_key,
    )
    slm = SLMPlane(slm_shape, wavelength_nm, slm_pixel_pitch_um, superpixel_size, input_amplitude, input_phase, initial_hologram)
    camera = CameraPlane(camera_shape, wavelength_nm, (1.0, 1.0), camera_pixel_pitch_um, np.zeros(camera_shape), np.zeros(camera_shape))
    propagator = Propagator(slm, camera, focal_length_mm, padding_factor)
    ideal_target = build_rectangle_target(camera_shape, propagator.camera_plane.x_axis_um, propagator.camera_plane.y_axis_um, rectangle_width_x_um, rectangle_width_y_um)
    target_amplitude = apply_psf_smoothing(ideal_target, psf_sigma_x_um, psf_sigma_y_um, propagator.camera_plane.scale_um)
    target_phase = np.zeros_like(target_amplitude)
    mask_center_x_um = 0.5 * (propagator.camera_plane.x_axis_um[0] + propagator.camera_plane.x_axis_um[-1])
    mask_center_y_um = 0.5 * (propagator.camera_plane.y_axis_um[0] + propagator.camera_plane.y_axis_um[-1])
    mask_radius_um = max(rectangle_width_x_um, rectangle_width_y_um) / 2.0 + mask_radius_margin_um
    target_mask = build_circular_mask(camera_shape, propagator.camera_plane.x_axis_um, propagator.camera_plane.y_axis_um, mask_center_x_um, mask_center_y_um, mask_radius_um)
    target_plane = CameraPlane(camera_shape, wavelength_nm, propagator.camera_plane.scale_um, camera_pixel_pitch_um, target_amplitude, target_phase)
    return propagator, target_plane, target_mask, input_amplitude, input_phase, calibration_results


initial_hologram = curvature_hologram(
    shape=slm_shape,
    linear_tilt=initial_phase_linear_tilt,
    astigmatism_weight=initial_phase_astigmatism_weight,
    quadratic_curvature=initial_phase_quadratic_curvature,
    linear_angle_rad=initial_phase_linear_angle_rad,
    conical_weight=initial_phase_conical_weight,
    center=(slm_shape[0] / 2.0, slm_shape[1] / 2.0),
)

propagator_1, target_plane_1, target_mask_1, input_amplitude_1, input_phase_1, calibration_results_1 = build_channel(wavelength_1_nm, slmsuite_calibration_h5_path_1)
propagator_2, target_plane_2, target_mask_2, input_amplitude_2, input_phase_2, calibration_results_2 = build_channel(wavelength_2_nm, slmsuite_calibration_h5_path_2)
optimizer = DWCGOptimizer(propagator_1, propagator_2, target_plane_1, target_mask_1, target_plane_2, target_mask_2)
optimizer.set_initial_hologram_array(initial_hologram)

print("Shared SLM shape:", slm_shape)
print("Channel 1 wavelength:", wavelength_1_nm, "nm")
print("Channel 2 wavelength:", wavelength_2_nm, "nm")
print("Channel 1 target active pixels:", int(np.sum(target_mask_1 > 0)))
print("Channel 2 target active pixels:", int(np.sum(target_mask_2 > 0)))

fig, axes = plt.subplots(2, 3, figsize=(13.5, 7.6), constrained_layout=True)
axes[0, 0].imshow(input_amplitude_1**2, origin="lower", cmap="magma", aspect="equal")
axes[0, 0].set_title("420 nm input intensity")
axes[0, 1].imshow(input_phase_1, origin="lower", cmap="twilight", aspect="equal")
axes[0, 1].set_title("420 nm input phase")
axes[0, 2].imshow(target_plane_1.amplitude**2 * target_mask_1, origin="lower", cmap="inferno", aspect="equal")
axes[0, 2].set_title("420 nm target and mask")
axes[1, 0].imshow(input_amplitude_2**2, origin="lower", cmap="magma", aspect="equal")
axes[1, 0].set_title("456 nm input intensity")
axes[1, 1].imshow(input_phase_2, origin="lower", cmap="twilight", aspect="equal")
axes[1, 1].set_title("456 nm input phase")
axes[1, 2].imshow(target_plane_2.amplitude**2 * target_mask_2, origin="lower", cmap="inferno", aspect="equal")
axes[1, 2].set_title("456 nm target and mask")
plt.show()


## 3. Optimize

In [ ]:
optimizer.optimize(
    maxiter=optimizer_maxiter,
    loss_scale=loss_scale,
    optimize_phase=optimize_phase,
    method=loss_method,
    channel_weight=channel_weight,
    exponential_rate=exponential_rate,
)

print("Dual-wavelength optimization finished")
print("Loss evaluations:", len(optimizer.loss_history))
print("Accepted CG iterations:", len(optimizer.iteration_loss_history))


## 4. Show Result

In [ ]:
result = optimizer.get_result_summary()
print("Channel 1 overlap:", result.channel_1.overlap)
print("Channel 1 efficiency:", result.channel_1.efficiency)
print("Channel 1 RMS error:", result.channel_1.rms_error)
print("Channel 1 phase error:", result.channel_1.phase_error)
print("Channel 2 overlap:", result.channel_2.overlap)
print("Channel 2 efficiency:", result.channel_2.efficiency)
print("Channel 2 RMS error:", result.channel_2.rms_error)
print("Channel 2 phase error:", result.channel_2.phase_error)
print("Optimization time (s):", result.optimization_time_sec)

optimizer.plot_result_summary()
plt.show()


## 5. Fit 456 nm Experimental Slope

Load a measured 456 nm line profile and fit intensity as a function of focal-plane length.


In [ ]:
# CSV/TSV file with columns: Length(um), Intensity
slope_profile_csv_path = "456nm_profile.csv"
tophat_threshold_fraction = 0.5
platform_window_threshold = 0.85


def load_profile_csv(profile_csv_path):
    try:
        data = np.genfromtxt(profile_csv_path, names=True, delimiter=None, dtype=float, encoding=None)
    except ValueError:
        data = np.genfromtxt(profile_csv_path, names=True, delimiter=",", dtype=float, encoding=None)
    if data.dtype.names is None or len(data.dtype.names) < 2:
        raise ValueError("Profile file must contain at least two named columns: Length(um) and Intensity.")
    length_name = data.dtype.names[0]
    intensity_name = data.dtype.names[1]
    length_um = np.asarray(data[length_name], dtype=float)
    intensity = np.asarray(data[intensity_name], dtype=float)
    finite = np.isfinite(length_um) & np.isfinite(intensity)
    if np.sum(finite) < 6:
        raise ValueError("Profile file must contain at least six finite data points for top-hat fitting.")
    order = np.argsort(length_um[finite])
    return length_um[finite][order], intensity[finite][order], length_name, intensity_name


def smooth_tophat_window(length_um, center_um, width_um, edge_um):
    length = np.asarray(length_um, dtype=float)
    edge = max(float(edge_um), np.finfo(float).eps)
    left_edge = float(center_um) - float(width_um) / 2.0
    right_edge = float(center_um) + float(width_um) / 2.0
    return 0.5 * (np.tanh((length - left_edge) / edge) - np.tanh((length - right_edge) / edge))


def sloped_tophat_model(length_um, background, center_intensity, slope, center_um, width_um, edge_um):
    window = smooth_tophat_window(length_um, center_um, width_um, edge_um)
    plateau = center_intensity + slope * (np.asarray(length_um, dtype=float) - center_um)
    return background + window * plateau


def initial_tophat_guess(length_um, intensity, threshold_fraction):
    length = np.asarray(length_um, dtype=float)
    signal = np.asarray(intensity, dtype=float)
    baseline = float(np.percentile(signal, 5.0))
    peak = float(np.max(signal))
    if peak <= baseline:
        raise ValueError("Cannot fit top-hat because profile peak is not above baseline.")
    threshold = baseline + threshold_fraction * (peak - baseline)
    active_indices = np.flatnonzero(signal >= threshold)
    if active_indices.size < 4:
        raise ValueError("Top-hat support has too few points. Lower tophat_threshold_fraction.")
    left_edge_um = float(length[int(active_indices[0])])
    right_edge_um = float(length[int(active_indices[-1])])
    width_um = right_edge_um - left_edge_um
    if width_um <= 0:
        raise ValueError("Initial top-hat width must be positive.")
    center_um = 0.5 * (left_edge_um + right_edge_um)
    edge_um = max(width_um / 25.0, np.median(np.diff(length)) if length.size > 1 else 1.0)
    center_intensity = max(peak - baseline, np.finfo(float).eps)
    return baseline, center_intensity, 0.0, center_um, width_um, edge_um


def fit_sloped_tophat(length_um, intensity, threshold_fraction, platform_threshold):
    from scipy.optimize import curve_fit

    length = np.asarray(length_um, dtype=float)
    signal = np.asarray(intensity, dtype=float)
    initial = initial_tophat_guess(length, signal, threshold_fraction)
    x_span = float(np.max(length) - np.min(length))
    y_span = float(np.max(signal) - np.min(signal))
    if x_span <= 0 or y_span <= 0:
        raise ValueError("Profile length and intensity spans must be positive.")
    lower_bounds = [0.0, 0.0, -10.0 * y_span / x_span, float(np.min(length)), 0.05 * x_span, np.median(np.diff(length)) / 10.0]
    upper_bounds = [float(np.max(signal)), 2.0 * float(np.max(signal)), 10.0 * y_span / x_span, float(np.max(length)), x_span, 0.5 * x_span]
    params, covariance = curve_fit(
        sloped_tophat_model,
        length,
        signal,
        p0=initial,
        bounds=(lower_bounds, upper_bounds),
        maxfev=20000,
    )
    background, center_intensity, slope, center_um, width_um, edge_um = [float(value) for value in params]
    window = smooth_tophat_window(length, center_um, width_um, edge_um)
    platform_mask = window >= platform_threshold
    if np.sum(platform_mask) < 2:
        raise ValueError("Fitted top-hat platform has too few points. Lower platform_window_threshold.")
    model = sloped_tophat_model(length, *params)
    plateau_line = background + center_intensity + slope * (length - center_um)
    return {
        "background": background,
        "center_intensity": center_intensity,
        "slope": slope,
        "center_um": center_um,
        "width_um": width_um,
        "edge_um": edge_um,
        "model": model,
        "plateau_line": plateau_line,
        "window": window,
        "platform_mask": platform_mask,
        "covariance": covariance,
    }


profile_length_um, profile_intensity, profile_length_name, profile_intensity_name = load_profile_csv(slope_profile_csv_path)
tophat_fit = fit_sloped_tophat(
    profile_length_um,
    profile_intensity,
    tophat_threshold_fraction,
    platform_window_threshold,
)
slope_fit = tophat_fit["slope"]
fit_center_um = tophat_fit["center_um"]
fit_reference_intensity = tophat_fit["background"] + tophat_fit["center_intensity"]

print("Loaded profile columns:", profile_length_name, profile_intensity_name)
print("Fitted top-hat center/width/edge (um):", tophat_fit["center_um"], tophat_fit["width_um"], tophat_fit["edge_um"])
print("Fitted background:", tophat_fit["background"])
print("Fitted center intensity:", fit_reference_intensity)
print("Fitted platform slope:", slope_fit)

fig, ax = plt.subplots(1, 1, figsize=(6.8, 4.2), constrained_layout=True)
ax.plot(profile_length_um, profile_intensity, "o", label="measured")
ax.plot(profile_length_um, tophat_fit["model"], "-", label="sloped top-hat fit")
ax.plot(profile_length_um[tophat_fit["platform_mask"]], tophat_fit["plateau_line"][tophat_fit["platform_mask"]], "--", label="platform slope")
ax.axvline(tophat_fit["center_um"] - tophat_fit["width_um"] / 2.0, color="gray", lw=1.0, alpha=0.7)
ax.axvline(tophat_fit["center_um"] + tophat_fit["width_um"] / 2.0, color="gray", lw=1.0, alpha=0.7)
ax.set_xlabel("Length (um)")
ax.set_ylabel("Intensity")
ax.set_title("456 nm sloped top-hat fit")
ax.grid(True, alpha=0.25, linestyle="--")
ax.legend()
plt.show()


## 6. Second Optimization with 456 nm Tilt Compensation

Keep the original input beams, keep the 420 nm target unchanged, replace the 456 nm target with the opposite tilted target, and use the previous final hologram as the initial hologram.


In [ ]:
def build_opposite_tilt_target(base_target_amplitude, x_axis_um, fitted_slope, reference_x_um, reference_intensity):
    base_target = np.asarray(base_target_amplitude, dtype=float)
    if base_target.ndim != 2:
        raise ValueError(f"base_target_amplitude must be 2D, got shape {base_target.shape}.")
    x_axis = np.asarray(x_axis_um, dtype=float)
    if x_axis.ndim != 1 or x_axis.size != base_target.shape[1]:
        raise ValueError(f"x_axis_um must have length {base_target.shape[1]}, got shape {x_axis.shape}.")
    reference = float(reference_intensity)
    if not np.isfinite(reference) or abs(reference) <= np.finfo(float).eps:
        raise ValueError(f"Invalid fitted reference intensity {reference_intensity}.")
    centered_x_um = x_axis - np.mean(x_axis)
    normalized_slope = float(fitted_slope) / reference
    compensation_ramp = 1.0 - normalized_slope * centered_x_um
    compensation_ramp = np.clip(compensation_ramp, 0.0, None)
    if np.max(compensation_ramp) <= 0:
        raise ValueError("Tilt compensation ramp is zero everywhere. Check the fitted top-hat platform slope.")
    tilted_intensity = base_target**2 * compensation_ramp[np.newaxis, :]
    peak = float(np.max(tilted_intensity))
    if peak <= 0:
        raise ValueError("Tilted 456 nm target contains no positive values.")
    tilted_amplitude = np.sqrt(tilted_intensity / peak)
    return tilted_amplitude, compensation_ramp / np.max(compensation_ramp)


tilted_target_amplitude_2, compensation_ramp_2 = build_opposite_tilt_target(
    base_target_amplitude=target_plane_2.amplitude,
    x_axis_um=propagator_2.camera_plane.x_axis_um,
    fitted_slope=slope_fit,
    reference_x_um=fit_center_um,
    reference_intensity=fit_reference_intensity,
)
compensated_target_plane_2 = CameraPlane(
    shape=camera_shape,
    wavelength_nm=wavelength_2_nm,
    scale_um=propagator_2.camera_plane.scale_um,
    camera_pixel_pitch_um=camera_pixel_pitch_um,
    amplitude=tilted_target_amplitude_2,
    phase=target_plane_2.phase,
)

optimizer_compensated = DWCGOptimizer(
    propagator_1=propagator_1,
    propagator_2=propagator_2,
    target_plane_1=target_plane_1,
    target_mask_1=target_mask_1,
    target_plane_2=compensated_target_plane_2,
    target_mask_2=target_mask_2,
)
optimizer_compensated.set_initial_hologram_array(optimizer.get_final_hologram())
optimizer_compensated.optimize(
    maxiter=optimizer_maxiter,
    loss_scale=loss_scale,
    optimize_phase=optimize_phase,
    method=loss_method,
    channel_weight=channel_weight,
    exponential_rate=exponential_rate,
)
compensated_result = optimizer_compensated.get_result_summary()

print("Compensated optimization finished")
print("Channel 1 overlap:", compensated_result.channel_1.overlap)
print("Channel 2 overlap:", compensated_result.channel_2.overlap)
print("Channel 2 efficiency:", compensated_result.channel_2.efficiency)

fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.0), constrained_layout=True)
axes[0].plot(propagator_2.camera_plane.x_axis_um, compensation_ramp_2)
axes[0].set_title("456 nm compensation ramp")
axes[0].set_xlabel("x (um)")
axes[0].set_ylabel("normalized ramp")
axes[0].grid(True, alpha=0.25, linestyle="--")
axes[1].imshow(target_plane_2.amplitude**2 * target_mask_2, origin="lower", cmap="inferno", aspect="equal")
axes[1].set_title("Original 456 nm target")
axes[2].imshow(compensated_target_plane_2.amplitude**2 * target_mask_2, origin="lower", cmap="inferno", aspect="equal")
axes[2].set_title("Tilt-compensated 456 nm target")
plt.show()

optimizer_compensated.plot_result_summary()
plt.show()
